1. Profile dataloader with multiple workers. My intuition is that since packing (via the sequence and packing iterator) is easier now, the bulk of time get spent in the preprocess iterator.

2. If the above is true(-ish), try caching patches instead of entropies. This will void the preprocess iterator latency.

3. If it's not true, see how efficient it is to simply pad instead of pack sequences (avoiding sequence iterator and packing iterator logic)

In [1]:
import sys; sys.path.append('/Users/arnavshah/Code/dnaBLT/training/data/iterators')
from training.data.iterators.v_args import TrainArgs
from training.data.iterators.v_arrow_iterator import ArrowFileIterator
from training.data.iterators.v_preprocess_iterator import PreprocessIterator

train_args = TrainArgs()
# train_args.data.buffer_size = 64
dataloader = train_args.data.build_from_rank(0, 1, 0, 1, "train")

In [2]:
import cProfile
import io
import pstats
from typing import Any, Dict, Generator

def profile_dataloader(dataloader: Any, num_iters: int = 10) -> Dict[str, Any]:
    """
    Profile a dataloader for a specified number of iterations using cProfile.
    
    Args:
        dataloader: The dataloader to profile
        num_iters: Number of iterations to run the profiler for
        
    Returns:
        Dictionary containing profiling results and statistics
    """
    pr = cProfile.Profile()
    pr.enable()
    
    # Run the dataloader for specified iterations
    batch_count = 0
    for i, batch in enumerate(dataloader):
        print(batch)
        if i >= num_iters:
            break
        batch_count += 1

    pr.disable()
    
    # Get stats
    s = io.StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats()
    
    stats = {
        'total_batches': batch_count,
        'profile_output': s.getvalue(),
        'stats': ps.stats,
        'total_time': ps.total_tt
    }
    
    return stats

    
# Run profiling
results = profile_dataloader(dataloader)

# Print results
print(f"Profiled {results['total_batches']} batches")
print("\nProfile output:")
print(results['profile_output'])
print(f"\nTotal time: {results['total_time']:.2f} seconds")
print(f"Time per batch: {results['total_time']/results['total_batches']*1000:.2f} ms")

Batch(x=tensor([[71, 75, 71,  ..., 71, 75, 75],
        [75, 88, 71,  ..., 71, 88, 88],
        [69, 75, 71,  ..., 71, 75, 71],
        ...,
        [75, 71, 71,  ..., 75, 88, 69],
        [88, 88, 71,  ..., 88, 71, 69],
        [69, 69, 88,  ..., 71, 75, 75]]), y=tensor([[75, 71, 71,  ..., 75, 75,  2],
        [88, 71, 69,  ..., 88, 88, 75],
        [75, 71, 71,  ..., 75, 71, 71],
        ...,
        [71, 71, 88,  ..., 88, 69, 88],
        [88, 71, 69,  ..., 71, 69, 71],
        [69, 88, 71,  ..., 75, 75, 88]]), mask=tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]]), patch_lengths=tensor([[ 1,  1,  1,  ...,  0,  0,  0],
        [ 1,  1,  1,  ...,  0,  0,  0],
        [ 1,  1,  1,  ...,  0,  0,  0],
        ...,
 

In [ ]:
preprocess_iterator = dataloader.sequence_iterator.source_to_iterator['16b*']._src_iter
batch = next(iter(preprocess_iterator))

In [ ]:
batch = next(iter(dataloader))
batch

In [ ]:
(batch.patch_lengths != 0).sum(dim=1)

# ~~0. Arrow iterator~~
# ~~1. Preprocess iterator~~
# ~~2. Sequence iterator~~
# ~~3. Packing iterator~~


# Transformer -> Attention + FFW

In [ ]:
batch.patch_lengths[0, :1609]

In [ ]:
iterator = iter(dataloader)
for _ in range(500):
    next(iterator)

In [ ]:
(500 * 16 * 4096) / sum(dataloader.sequence_iterator.source_to_iterator['16b*']._src_iter.patch_lengths)

In [ ]:
dataloader.sequence_iterator.source_to_iterator['16b*']._src_iter.arrow_batch_iterator.current_batch_idx

In [ ]:
import torch
import numpy as np
from tqdm import trange

all_nonzero_patch_lengths = []
length_sum = 0
patch_sum = 0

for _ in trange(1000):
    big_batch = next(preprocess_iterator)
    patch_lengths = big_batch.patch_lengths
    length_sum += (patch_lengths != 0).sum()
    patch_sum += patch_lengths.sum()
    # Flatten, filter, convert to numpy, and append
    nonzero_patch_lengths = patch_lengths[patch_lengths != 0]
    all_nonzero_patch_lengths.append(nonzero_patch_lengths.cpu())  # ensure on CPU if tensor

print("Average patch size", patch_sum / length_sum)
# Concatenate all batches into a single tensor
all_nonzero_patch_lengths = torch.cat(all_nonzero_patch_lengths, dim=0)

# Now proceed with your logic
tensor_np = all_nonzero_patch_lengths.numpy()
max_patch = all_nonzero_patch_lengths.max().item()
value_range = np.arange(1, max_patch + 1)
counts = np.bincount(tensor_np, minlength=max_patch + 1)[1:]  # skip index 0
weighted = counts * value_range
cdf = weighted.cumsum() / weighted.sum()
idx_99 = (cdf > 0.99).argmax()

print(f"99% of the weighted patch length mass is covered by patch length: {value_range[idx_99]}")